In [1]:
from openai import OpenAI
BASE_URL = "https://api.gapgpt.app/v1"
API_KEY  = "sk-s8KnoW59PPxeHBvyzENeVoEiH2QbiNm1PxJt20H586up5p8n"

client_openai = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams , SparseVectorParams

client = QdrantClient(url="http://localhost:6333")



client.recreate_collection(
    collection_name="schema_docs_hybrid",
    vectors_config={
        "dense": VectorParams(size=3072, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

/tmp/ipykernel_85165/2698050574.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [4]:
customers_text = """Table: sales.customers

نام جدول:
مشتریان

توضیحات:
اطلاعات مشتریان فروشگاه و سوابق خرید آن‌ها.

هدف تجاری:
برای شناسایی و تحلیل مشتریان، ارزیابی ارزش مشتریان، بررسی وفاداری مشتریان، تحلیل الگوهای خرید و تهیه گزارش‌های فروش مبتنی بر مشتری استفاده می‌شود.

کلیدواژه‌ها:
مشتری
خریدار
مشتری فعال
مشتری جدید
مشتری وفادار
مشتری برتر
رفتار خرید
تحلیل مشتریان
خرید مشتری
درآمد مشتری
ارزش مشتری
سابقه خرید

سوالات رایج:
- تعداد مشتریان چقدر است؟
- مشتریان برتر چه کسانی هستند؟
- مشتریان فعال چه کسانی هستند؟
- مشتریان جدید چه کسانی هستند؟
- مجموع خرید هر مشتری چقدر است؟
- میانگین خرید هر مشتری چقدر است؟
- هر مشتری چند سفارش ثبت کرده است؟
- کدام مشتری بیشترین خرید را داشته است؟
- کدام مشتری بیشترین درآمد را ایجاد کرده است؟
- آخرین خرید هر مشتری چه زمانی بوده است؟
- مشتریان هر شهر چه تعداد هستند؟
- مشتریان هر استان چه تعداد هستند؟"""

In [5]:
orders_text = """Table: sales.orders

نام جدول:
سفارش‌ها

توضیحات:
اطلاعات سفارش‌های خرید مشتریان و وضعیت پردازش آن‌ها را نگهداری می‌کند.

هدف تجاری:
برای تحلیل فروش، بررسی روند سفارشات، ارزیابی عملکرد فروشگاه، اندازه‌گیری درآمد، پایش وضعیت سفارش‌ها و تهیه گزارش‌های فروش استفاده می‌شود.

کلیدواژه‌ها:
سفارش
سفارش مشتری
خرید
فروش
تراکنش فروش
ثبت سفارش
وضعیت سفارش
سفارش تکمیل شده
سفارش لغو شده
سفارش در حال پردازش
تعداد سفارشات
روند فروش
درآمد فروش
فروش دوره‌ای
تاریخ سفارش
محاسبه سود وزیان

سوالات رایج:

* تعداد کل سفارشات چقدر است؟
* در ماه گذشته چند سفارش ثبت شده است؟
* در بازه زمانی مشخص چند سفارش ثبت شده است؟
* روند فروش ماهانه چگونه بوده است؟
* روند سفارشات ماهانه چگونه بوده است؟
* هر مشتری چه سفارش‌هایی ثبت کرده است؟
* کدام مشتری بیشترین تعداد سفارش را ثبت کرده است؟
* میزان فروش در هر بازه زمانی چقدر بوده است؟
* بیشترین درآمد در چه دوره‌ای ایجاد شده است؟
* تعداد سفارشات هر فروشگاه چقدر است؟
* هر کارمند چند سفارش را مدیریت کرده است؟
* سفارشات تکمیل شده چه تعداد هستند؟
* سفارشات لغو شده چه تعداد هستند؟
* سفارشات در حال پردازش چه تعداد هستند؟
* کدام سفارش‌ها با تأخیر ارسال شده‌اند؟
* میانگین تعداد سفارشات در هر ماه چقدر است؟
* آخرین سفارش‌های ثبت شده کدام هستند؟
* بیشترین فروش مربوط به چه بازه زمانی است؟
  """


In [6]:
order_items_text = """Table: sales.order_items

نام جدول:
اقلام سفارش

توضیحات:
جزئیات محصولات موجود در سفارش‌های مشتریان را نگهداری می‌کند و ارتباط بین سفارش‌ها و محصولات فروخته‌شده را نمایش می‌دهد.

هدف تجاری:
برای تحلیل فروش محصولات، محاسبه درآمد، بررسی حجم فروش، ارزیابی عملکرد کالاها، تحلیل تخفیف‌ها و شناسایی محصولات پرفروش و کم‌فروش استفاده می‌شود.

کلیدواژه‌ها:
اقلام سفارش
جزئیات سفارش
محصولات سفارش
محصولات فروخته شده
فروش محصول
فروش کالا
درآمد محصول
درآمد کالا
مقدار فروش
تعداد فروش
حجم فروش
مقدار خرید
تعداد خرید
حجم خرید
تخفیف
تخفیف محصول
پرفروش‌ترین محصول
کم‌فروش‌ترین محصول
عملکرد محصول
عملکرد کالا
سهم فروش محصول
خرید مشتریان
سوداور
زیان اور
بازده

سوالات رایج:

* پرفروش‌ترین محصولات کدام‌اند؟
* کدام محصول بیشترین فروش را داشته است؟
* کدام محصول بیشترین درآمد را ایجاد کرده است؟
* درآمد حاصل از هر محصول چقدر است؟
* مجموع تعداد محصولات فروخته شده چقدر است؟
* چه میزان تخفیف روی محصولات اعمال شده است؟
* میزان فروش هر محصول چقدر است؟
* رتبه‌بندی محصولات بر اساس فروش چگونه است؟
* کدام محصولات کمترین فروش را داشته‌اند؟
* سهم هر محصول از درآمد کل چقدر است؟
* در یک بازه زمانی مشخص چه تعداد از هر محصول فروخته شده است؟
* میانگین فروش هر محصول چقدر است؟
* بیشترین تعداد فروش مربوط به کدام محصول است؟
* کدام محصولات بیشترین تخفیف را داشته‌اند؟
* درآمد هر دسته از محصولات چقدر است؟
* سهم هر محصول از کل فروش چقدر است؟
* روند فروش محصولات در طول زمان چگونه بوده است؟
* کدام کالاها بیشترین حجم فروش را داشته‌اند؟
* هر مشتری چه سفارش‌هایی ثبت کرده است؟
* کدام مشتری بیشترین تعداد سفارش را ثبت کرده است؟
  """


In [7]:
products_text = """Table: production.products

نام جدول:
محصولات

توضیحات:
اطلاعات محصولات و کالاهای قابل فروش را نگهداری می‌کند.

هدف تجاری:
برای مدیریت محصولات، دسته‌بندی کالاها، قیمت‌گذاری، مدیریت سبد محصولات و تحلیل ویژگی‌های محصولات استفاده می‌شود.

کلیدواژه‌ها:
محصول
کالا
آیتم
کالای فروشگاهی
فهرست محصولات
محصولات فروشگاه
کاتالوگ محصولات
دسته‌بندی محصول
برند محصول
قیمت محصول
مشخصات محصول

سوالات رایج:

* چه محصولاتی در سیستم ثبت شده‌اند؟
* قیمت هر محصول چقدر است؟
* گران‌ترین محصولات کدام‌اند؟
* ارزان‌ترین محصولات کدام‌اند؟
* میانگین قیمت محصولات چقدر است؟
* محصولات هر دسته‌بندی کدام‌اند؟
* محصولات هر برند کدام‌اند؟
* تعداد محصولات چقدر است؟
* جدیدترین محصولات کدام‌اند؟
* هر دسته‌بندی چند محصول دارد؟
* هر برند چند محصول دارد؟
* مشخصات هر محصول چیست؟
* محصولات فعال کدام‌اند؟
* محصولات غیرفعال کدام‌اند؟
  """


In [8]:
brands_text = """Table: production.brands

نام جدول:
برندها

توضیحات:
اطلاعات برندها و تولیدکنندگان محصولات را نگهداری می‌کند.

هدف تجاری:
برای تحلیل عملکرد برندها، مقایسه برندهای مختلف، ارزیابی سهم برندها از فروش و بررسی تنوع محصولات هر برند استفاده می‌شود.

کلیدواژه‌ها:
برند
نام تجاری
تولیدکننده
شرکت تولیدکننده
برند محصول
برند کالا
محصولات برند
عملکرد برند
فروش برند
درآمد برند
سهم برند
برند پرفروش

سوالات رایج:

* پرفروش‌ترین برند کدام است؟
* کدام برند بیشترین درآمد را ایجاد کرده است؟
* میزان فروش هر برند چقدر است؟
* عملکرد برندهای مختلف چگونه است؟
* محصولات هر برند کدام‌اند؟
* هر برند چند محصول دارد؟
* سهم هر برند از فروش کل چقدر است؟
* رتبه‌بندی برندها بر اساس فروش چگونه است؟
* کدام برند کمترین فروش را داشته است؟
* میانگین فروش محصولات هر برند چقدر است؟
* کدام برند بیشترین تعداد محصول را دارد؟
* کدام برند کمترین تعداد محصول را دارد؟
  """


In [9]:
categories_text = """Table: production.categories

نام جدول:
دسته‌بندی محصولات

توضیحات:
اطلاعات دسته‌بندی‌ها و گروه‌های محصولات را نگهداری می‌کند.

هدف تجاری:
برای سازمان‌دهی محصولات، گروه‌بندی کالاها، مدیریت ساختار محصولات و تحلیل محصولات بر اساس دسته‌بندی استفاده می‌شود.

کلیدواژه‌ها:
دسته‌بندی
گروه محصول
طبقه‌بندی محصولات
دسته محصولات
گروه‌بندی کالاها
دسته کالا
گروه کالا
دسته محصول
شاخه محصول
رده محصول

سوالات رایج:

* چه دسته‌بندی‌هایی در سیستم وجود دارد؟
* هر دسته‌بندی شامل چه محصولاتی است؟
* هر دسته‌بندی چند محصول دارد؟
* محصولات هر دسته‌بندی کدام‌اند؟
* تعداد دسته‌بندی‌ها چقدر است؟
* بزرگ‌ترین دسته‌بندی کدام است؟
* کوچک‌ترین دسته‌بندی کدام است؟
* محصولات یک دسته‌بندی خاص کدام‌اند؟
* کدام دسته‌بندی بیشترین تعداد محصول را دارد؟
* کدام دسته‌بندی کمترین تعداد محصول را دارد؟
  """


In [10]:
stocks_text = """Table: production.stocks

نام جدول:
موجودی کالا

توضیحات:
اطلاعات موجودی محصولات را به تفکیک فروشگاه یا انبار نگهداری می‌کند.

هدف تجاری:
برای مدیریت موجودی کالا، کنترل سطح موجودی، جلوگیری از کمبود کالا، برنامه‌ریزی تأمین و پایش وضعیت انبار استفاده می‌شود.

کلیدواژه‌ها:
موجودی
انبار
موجودی کالا
موجودی محصول
موجودی محصولات
کنترل موجودی
مدیریت انبار
سطح موجودی
کسری موجودی
کمبود کالا
تأمین کالا
ناموجودی
دسترس‌پذیری کالا
موجودی فروشگاه

سوالات رایج:

* موجودی هر محصول چقدر است؟
* موجودی هر محصول در هر فروشگاه چقدر است؟
* موجودی کل هر محصول چقدر است؟
* کدام محصولات موجودی کمی دارند؟
* کدام محصولات در آستانه اتمام موجودی هستند؟
* چه کالاهایی نیاز به تأمین مجدد دارند؟
* کدام محصولات ناموجود هستند؟
* کدام فروشگاه‌ها کمبود موجودی دارند؟
* کدام محصولات بیشترین موجودی را دارند؟
* وضعیت موجودی کالاها چگونه است؟
* چه مقدار موجودی از هر کالا در انبارها وجود دارد؟
* موجودی یک محصول خاص چقدر است؟
* چه کالاهایی در انبار موجود نیستند؟
  """


In [11]:
stores_text = """Table: sales.stores

نام جدول:
فروشگاه‌ها

توضیحات:
اطلاعات فروشگاه‌ها، شعب و مراکز فروش را نگهداری می‌کند.

هدف تجاری:
برای تحلیل عملکرد شعب، مقایسه فروشگاه‌ها، ارزیابی درآمد مراکز فروش، بررسی پوشش جغرافیایی فروش و تهیه گزارش‌های عملکرد شعب استفاده می‌شود.

کلیدواژه‌ها:
فروشگاه
شعبه
مرکز فروش
شعب فروش
فروشگاه زنجیره‌ای
شعبه فروش
موقعیت فروشگاه
شهر فروشگاه
منطقه فروشگاه
عملکرد شعب
عملکرد فروشگاه
فروش شعب
درآمد شعب
فروشگاه پرفروش
شعبه پرفروش

سوالات رایج:

* چه فروشگاه‌هایی در سیستم ثبت شده‌اند؟
* تعداد فروشگاه‌ها چقدر است؟
* میزان فروش هر فروشگاه چقدر است؟
* کدام فروشگاه بیشترین فروش را داشته است؟
* کدام فروشگاه بیشترین درآمد را ایجاد کرده است؟
* کدام فروشگاه بهترین عملکرد را داشته است؟
* فروش شعب مختلف چگونه با هم مقایسه می‌شود؟
* رتبه‌بندی فروشگاه‌ها بر اساس فروش چگونه است؟
* سهم هر فروشگاه از فروش کل چقدر است؟
* عملکرد شعب مختلف چگونه است؟
* فروش هر شعبه در طول زمان چگونه تغییر کرده است؟
* کدام شهر بیشترین فروش را داشته است؟
* کدام منطقه بیشترین فروش را داشته است؟
* هر فروشگاه چند سفارش داشته است؟
* هر فروشگاه به چند مشتری خدمت‌رسانی کرده است؟
  """

In [12]:
staffs_text = """Table: sales.staffs

نام جدول:
کارکنان

توضیحات:
اطلاعات کارکنان فروشگاه‌ها، فروشندگان و مدیران را نگهداری می‌کند.

هدف تجاری:
برای ارزیابی عملکرد کارکنان، تحلیل عملکرد فروشندگان، انتساب فروش به هر کارمند، بررسی بهره‌وری تیم فروش و تهیه گزارش‌های عملکرد پرسنل استفاده می‌شود.

کلیدواژه‌ها:
کارمند
پرسنل
فروشنده
کارشناس فروش
نیروی فروش
مدیر
مدیر فروشگاه
مدیر شعبه
کارکنان فروش
تیم فروش
عملکرد کارکنان
عملکرد فروشندگان
فروشنده برتر
بهره‌وری کارکنان
درآمد کارکنان

سوالات رایج:

* چه کارکنانی در سیستم ثبت شده‌اند؟
* تعداد کارکنان چقدر است؟
* بهترین فروشندگان کدام‌اند؟
* کدام کارمند بیشترین فروش را داشته است؟
* کدام کارمند بیشترین درآمد را ایجاد کرده است؟
* عملکرد هر کارمند چگونه است؟
* رتبه‌بندی کارکنان بر اساس فروش چگونه است؟
* هر کارمند چه میزان فروش داشته است؟
* هر کارمند چند سفارش را مدیریت کرده است؟
* کدام مدیر فروش عملکرد بهتری دارد؟
* بهره‌وری هر فروشنده چگونه است؟
* عملکرد کارکنان هر فروشگاه چگونه است؟
* کدام کارکنان بیشترین تعداد سفارش را ثبت کرده‌اند؟
* میانگین فروش هر کارمند چقدر است؟
* سهم هر کارمند از فروش کل چقدر است؟
  """

In [13]:
docs = [
    {"table": "sales.customers", "text": customers_text},
    {"table": "sales.orders", "text": orders_text},
    {"table": "sales.order_items", "text": order_items_text},
    {"table": "production.products", "text": products_text},
    {"table": "production.brands", "text": brands_text},
    {"table": "production.categories", "text": categories_text},
    {"table": "production.stocks", "text": stocks_text},
    {"table": "sales.stores", "text": stores_text},
    {"table": "sales.staffs", "text": staffs_text}
]

In [27]:
from collections import defaultdict
import math

def tokenize(text):
    return text.lower().split()

vocab = {}
df = defaultdict(int)
doc_lens = []

for doc in docs:
    tokens = tokenize(doc["text"])
    doc_lens.append(len(tokens))
    unique_tokens = set(tokens)
    for t in unique_tokens:
        df[t] += 1

for i, token in enumerate(df.keys()):
    vocab[token] = i

N = len(docs)
avgdl = sum(doc_lens) / len(doc_lens)  # ✅ میانگین به جای max
K1 = 1.5
B = 0.75


def bm25_sparse(text, is_query=False):
    tokens = tokenize(text)
    tf = defaultdict(int)
    for t in tokens:
        tf[t] += 1

    doc_len = len(tokens)
    indices = []
    values = []

    for token, freq in tf.items():
        if token not in vocab:
            continue  # ✅ توکن‌های ناشناخته (مهم برای query) رد می‌شوند

        idf = math.log((N - df[token] + 0.5) / (df[token] + 0.5) + 1)

        if is_query:
            # ✅ برای query فقط IDF — وزن‌دهی سبک‌تر و استاندارد
            score = idf
        else:
            # ✅ فرمول BM25 کامل با avgdl
            tf_norm = (freq * (K1 + 1)) / (freq + K1 * (1 - B + B * doc_len / avgdl))
            score = idf * tf_norm

        indices.append(vocab[token])
        values.append(float(score))  # ✅ اطمینان از float برای qdrant

    return {
        "indices": indices,
        "values": values
    }

In [28]:
from openai import OpenAI
BASE_URL = "https://api.gapgpt.app/v1"
API_KEY  = "sk-s8KnoW59PPxeHBvyzENeVoEiH2QbiNm1PxJt20H586up5p8n"

client_openai = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)
vectors = []

In [ ]:
for doc in docs:
    dense = client_openai.embeddings.create(
        model="text-embedding-3-large",
        input=doc["text"]
    ).data[0].embedding

    sparse = bm25_sparse(doc["text"])

    vectors.append({
        "table": doc["table"],
        "text": doc["text"],
        "dense": dense,
        "sparse": sparse
    })

In [ ]:
from qdrant_client.models import PointStruct, SparseVector
import uuid

points = []

for i, vec in enumerate(vectors):
    point = PointStruct(
        id=i,
        vector={
            "dense": vec["dense"],
            "sparse": SparseVector(
                indices=vec["sparse"]["indices"],
                values=vec["sparse"]["values"]
            )
        },
        payload={
            "table": vec["table"],
            "text": vec["text"]
        }
    )
    points.append(point)

client.upsert(
    collection_name="schema_docs_hybrid",
    points=points
)

print(f"{len(points)} سند با موفقیت ذخیره شد")

✅ 9 سند با موفقیت ذخیره شد


In [46]:
question = "پرفروش ترین محصول کدام است ؟"

In [47]:
dense_query = client_openai.embeddings.create(
    model="text-embedding-3-large",
    input=question
).data[0].embedding

In [48]:
sparse_query = bm25_sparse(question)

In [49]:
from qdrant_client.models import (
    Prefetch,
    FusionQuery,
    Fusion,
    SparseVector
)

results = client.query_points(
    collection_name="schema_docs_hybrid",
    prefetch=[
        Prefetch(
            query=dense_query,
            using="dense",
            limit=10,
        ),
        Prefetch(
            query=SparseVector(
                indices=sparse_query["indices"],
                values=sparse_query["values"]
            ),
            using="sparse",
            limit=10,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
)

In [50]:
for p in results.points:
    print(
        f"{p.payload['table']} | score={round(p.score,4)}"
    )

production.brands | score=0.7
production.products | score=0.6667
sales.order_items | score=0.5833
sales.orders | score=0.4444
sales.stores | score=0.3611
